# A marketing mix model, sampled in your browser

This notebook uses **nuts-rs-wasm**, exactly like the interactive demo. Python in this notebook only prepares the model source and displays the browser view. Numba compilation, Rust NUTS, parameter expansion and Arrow storage all run in that view, on your device.

The example has 179 weeks and two channels. Edit the model source or turn yearly seasonality off below, run the cells, and click **Start sampling**. The first run loads about 120 MB of runtime assets. A local Jupyter notebook needs only IPython; PyMC is supplied by the browser runtime.


In [ ]:
model_source = """
import json

import arviz_stats as az
import numpy as np
import pandas as pd
from pymc_marketing.mmm import MMM, GeometricAdstock, LogisticSaturation

seasonality = globals().get("SEASONALITY", True)
data = pd.read_csv(
    globals().get("DATA_PATH", "/mmm_example.csv"), parse_dates=["date_week"]
)
mmm = MMM(
    date_column="date_week",
    channel_columns=["x1", "x2"],
    control_columns=["event_1", "event_2", "t"],
    adstock=GeometricAdstock(l_max=8),
    saturation=LogisticSaturation(),
    yearly_seasonality=2 if seasonality else None,
)
mmm.build_model(data.drop(columns="y"), data.y)
model = mmm._get_sampling_model()
var_names = [
    "adstock_alpha",
    "saturation_lam",
    "saturation_beta",
    "gamma_control",
    "y_sigma",
    "intercept_contribution",
]
if seasonality:
    var_names.append("gamma_fourier")


def emit(**event):
    print("MMM_EVENT " + json.dumps(event), flush=True)
"""


## Run the model

Use the hosted demo origin below, or extract both release archives and serve their combined directory locally (`python -m http.server 8000`), then set `demo_url = "http://localhost:8000"`. The configured model is passed in a URL fragment, which is not sent in HTTP requests. Only run model code you trust.


In [ ]:
from IPython.display import IFrame, display
from urllib.parse import quote
import json

demo_url = "https://pymc-labs.github.io/nuts-rs-wasm"
configuration = {
    "source": model_source,
    "seasonality": True,  # Change to False and rerun to compare.
    "options": {"chains": 2, "tune": 750, "draws": 500, "seed": 42},
}
url = demo_url + "/notebook.html#" + quote(json.dumps(configuration), safe="")
display(IFrame(url, width="100%", height=950))


## Keep the posterior

Download the four Arrow files from the completed run: one posterior and one sampler-statistics file per chain. The values are on the original parameter scale, including selected deterministics. The view reports R-hat, bulk/tail ESS and divergences. This two-chain run is a demonstration; increase the sampling budget and assess convergence before making decisions.

For local analysis with PyArrow installed:
```python
import pyarrow.ipc as ipc
posterior = ipc.open_stream("chain-1-posterior.arrow").read_all()
posterior.schema
```

[Adapter, build instructions and release assets](https://github.com/pymc-labs/nuts-rs-wasm)
